# An optical experiment guided by AI

**Goal:** find an input pulse that gives a smaller distortion score while keeping useful light.

We repeat four steps: **look at earlier results → choose a test → simulate the light → save and check the result.**
Codex chooses what to try and how much work to spend. Julia calculates what happens to the pulse.

The starting example matches `dio_power_optimization.ipynb`: a **200 fs Gaussian**, **LP01 launch**,
**three propagated modes**, **2.86 m fiber**, and **1379.2 nm carrier**.
We vary the input peak power, starting at **100 kW** within **50–150 kW**.

The score is **D_IO**, which compares the input pulse with the nonlinear output. Lower is better.
D_IO ignores overall amplitude scaling, so we also monitor the LP01 photon count.

Use **GNLSE (Python)** in VS Code. Run the cells from top to bottom.
The next cell only loads the helpers; you do not need to edit it.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
sys.path.insert(0, str(ROOT / "notebook_helpers" / "ai_augmented_loop"))
from notebook_tools import start_study

## 1. Choose the study

Choose the experiment and how much work to allow:

| Setting | Meaning |
|---|---|
| `EXAMPLE` | Which physical experiment to use. Leave it as `dio_power` for this study. |
| `BRIGHTNESS_TARGET` | Desired LP01 photon count, as a **percentage** of the reference. Write **50**, not 0.50. |
| `MAX_TESTS` | Total number of new simulation evaluations initially allowed. |
| `MAX_AI_CALLS` | Maximum AI call attempts, including retries when Codex is busy. |
| `MAX_MINUTES` | Time allowance for the AI planning and simulation work together. |

The brightness target is a preference: interesting results below it are kept.
For this D_IO example, the reference is the **100 kW nonlinear output**, calculated with 1 mm steps.

Saving and resuming happen automatically. If a saved study exists, its settings and remaining
allowances are used. For a fresh study, add `fresh=True` to `start_study(...)`.
To add budget to a saved study, use `study.allow_more(...)` in section 6.
The remaining amounts are printed below. There are no forms or JSON to type.

**The AI budget counts call attempts, including failed calls.** One successful call can
request several simulation tests.

In [ ]:
EXAMPLE = "dio_power"  # Choose the physical experiment.
BRIGHTNESS_TARGET = 50  # Aim to keep at least 50% of the reference photons.
MAX_TESTS = 80  # Initially allow up to 80 new simulation tests.
MAX_AI_CALLS = 10  # Initially allow ten AI call attempts, including retries.
MAX_MINUTES = 60  # Initially allow one hour of AI and simulation work.
study = start_study(EXAMPLE, brightness_percent=BRIGHTNESS_TARGET,  # Open or continue this study.
                    max_tests=MAX_TESTS, max_ai_calls=MAX_AI_CALLS,  # Set the simulation and AI budgets.
                    max_minutes=MAX_MINUTES)  # Set the time allowance for a new study.

## 2. Measure the starting pulse

Connect to your signed-in Codex and measure the baseline once. A saved baseline is reused.
Running this notebook calls Codex through your local sign-in; each planning call is separate
from this chat. If sign-in is needed, the notebook tells you how.

One test scores one launch. It can involve more than one light-propagation calculation,
because we also calculate reference or diagnostic fields when needed.

In [ ]:
study.prepare()  # Connect Codex and measure the starting pulse if needed.

## 3. Let AI choose what to try

Codex looks at the measured history and can also use general physics knowledge. It may
try another launch, take a few gradient-descent steps, check numerical accuracy, or explore
a wider range. **Three decisions do not mean three simulations:** one decision may request
several simulation tests.

The program can expand a power range by **25 kW** when an outward test supports improvement.
There is no preset upper-power ceiling; each accepted power adjustment is at most **10 kW**.
Codex may also revise a threshold. The previous value, new value and reason are saved.

The output below tells you **what Codex chose, why, and what the simulation found**.
Rerun this cell whenever you want another batch, within the remaining allowance.

**If Codex says its model is at capacity, the notebook waits and tries again:** first
15 seconds, then 30 seconds, then 60 seconds between attempts. Each attempt uses one
AI call; waiting uses the time allowance. Retries continue while those allowances remain.
Your measurements and gradient-descent path stay saved. Other errors are shown for you
to check. You can interrupt the cell at any time and resume later.

In [ ]:
study.run(decisions=3)  # Carry out up to three AI decisions; retry automatically if Codex is busy.

## 4. See whether the result improved

The table shows each tested launch, its score, and its brightness relative to the reference.
The plots show the tested points and accepted gradient-descent steps.

**Needs checking** means the result is still provisional. We keep it so a promising result
can receive more work. **Meets current checks** refers to the recorded thresholds; it does
not prove a global minimum.

In [ ]:
study.results()  # Show the saved measurements and optimization plots.

## 5. See what changed and inspect the pulse

The first command lists any expanded ranges and revised thresholds.
It reads the latest saved settings, including changes made since this notebook was opened.
The second shows **time and spectrum plots** for the first saved optical result and the
lowest recorded score at the search resolution. Each plot includes input, linear output,
and nonlinear output. These commands read saved results; they do not run new simulations.

In [ ]:
study.changes()  # Show range and threshold changes, if any.
study.pulses()  # Show the saved pulse shapes in time and wavelength.

## 6. Continue when useful

The next command shows **why the last batch stopped**, what its last action did,
and how many tests, minutes and AI call attempts remain. The reason is saved with
the study, so it is still available when you reopen it.

| Message | Meaning |
|---|---|
| Completed the requested AI decisions | This batch finished; more optimization may still be useful. |
| AI-call, time or simulation allowance used up | Add the corresponding allowance before continuing. |
| Codex requested a pause | Read its saved reason before deciding whether to continue. |
| AI call failed / interrupted | Read the error, or resume the saved work after an interruption. |
| Locally converged | The checked local slopes are small under the current settings. |
| Accuracy limited | Finer calculations are needed to trust the slopes. |
| No acceptable step | The attempted moves did not meet the current rules. |
| Budget or iteration allocation reached | The allowed work was used; this does not establish a minimum. |

Your completed work is saved. Rerun section 3 to continue if allowance remains.
To grant more work, remove the first `#` from one optional line below. Each execution adds
that allowance again. To add only AI calls, keep `tests=0` and `minutes=0`.

In [ ]:
study.remaining()  # Show why the last batch stopped and how much work remains.
# study.allow_more(tests=40, minutes=30, ai_calls=5)  # Add tests, time and five AI calls.
# study.allow_more(tests=0, minutes=0, ai_calls=5)  # Add only five AI calls.

## 7. Finish for now

This closes the idle simulator. Your measurements and progress remain saved, and the
simulator restarts automatically if you continue later.

In [ ]:
study.finish()  # Close the simulator for now; keep all saved results.

## Optional choices

The loop is general. Change `EXAMPLE` to use another starting experiment:

| Example | What changes |
|---|---|
| `dio_power` | Vary the LP01 input power and minimize input/output distortion D_IO. |
| `lp01_auxiliary_dnl` | Fix mode 1 at 100 kW; vary helper powers and phases; minimize LP01 D_NL. |
| `lp01_auxiliary_dio` | Use the same helper controls but minimize LP01 D_IO. |
| `demo` | Try the workflow on a fast mathematical example, not optical data. |

**D_NL compares linear and nonlinear outputs; D_IO compares input and nonlinear output.**
The auxiliary examples use matched linear LP01 photons as their brightness reference.

Other settings can be added to `start_study(...)` when needed: `fresh=True` for a new study,
or `brightness_required=True` to make the brightness target a requirement.
For an equipment-tolerance test, give ordinary numbers
such as `power_error_kW=1.0` and, for the phase examples, `phase_error_degrees=1.0`.
These are uncertainty bounds you choose, not assumed equipment specifications.
New starting settings apply to a fresh study; resuming keeps the saved settings.

The database and technical details are in [the usage guide](../../../notebook_helpers/README.md). The physics and
optimization code are the same as before; these short commands handle the bookkeeping.

**After helper-code updates:** reopen the saved notebook, restart its kernel, and run the
first two code cells. You can then view results and status without running new experiments.

**A short way to explain this notebook:**
“We let AI review earlier simulations and choose the next useful test. The numerical model
calculates the output pulse. We check and save that result, then repeat while improvement
looks promising and computation remains available.”